In [1]:
import cv2
import torch
import torch.nn as nn
import numpy as np
import os
import time
import pickle
import base64
import threading
import ollama

from ultralytics import YOLO
from pytorchvideo.models.hub import slowfast_r50

from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders


In [2]:
# -------------------------------------------------
# CONFIGURATION
# -------------------------------------------------

ALERT_EMAIL = "bilalkachij@gmail.com"

EVENT_COOLDOWN = 15
CLIP_LENGTH = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEMP_VIDEO_PATH = "event_clip.mp4"
VIDEO_FPS = 5

PROCESS_EVERY_N_FRAME = 5


clip_frames = []
last_alert_time = 0
video_path = None
fight_status = False

weapon_boxes = None
fire_boxes = None


# -------------------------------------------------
# LOAD MODELS
# -------------------------------------------------

print("🔄 Loading models...")

weapon_model = YOLO("models/weapon_detector.pt")
fire_model = YOLO("models/fire_detector.pt")

fight_detector = slowfast_r50(pretrained=False)

fight_detector.blocks[-1].proj = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(2304, 2)
)

fight_detector.load_state_dict(
    torch.load("models/fight_detector.pt", map_location=DEVICE)
)

fight_detector = fight_detector.to(DEVICE).eval()

print("✅ All models loaded")


# -------------------------------------------------
# GMAIL SERVICE
# -------------------------------------------------

SCOPES = ['https://www.googleapis.com/auth/gmail.send']

def gmail_service():

    creds = None

    if os.path.exists("token.pickle"):
        with open("token.pickle", "rb") as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:

        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())

        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)

        with open("token.pickle", "wb") as token:
            pickle.dump(creds, token)

    return build("gmail", "v1", credentials=creds)


# -------------------------------------------------
# EMAIL ALERT
# -------------------------------------------------

def send_email_async(subject, events, attachment_path):

    def email_task():

        try:

            prompt = f"Detected security events: {', '.join(events)}. Write a short emergency alert."

            response = ollama.chat(
                model="gpt-oss:120b-cloud",
                messages=[{"role": "user", "content": prompt}]
            )

            body = response["message"]["content"]

            service = gmail_service()

            message = MIMEMultipart()
            message["to"] = ALERT_EMAIL
            message["subject"] = subject

            message.attach(MIMEText(body, "plain"))

            if attachment_path and os.path.exists(attachment_path):

                part = MIMEBase("application", "octet-stream")

                with open(attachment_path, "rb") as f:
                    part.set_payload(f.read())

                encoders.encode_base64(part)

                part.add_header(
                    "Content-Disposition",
                    f"attachment; filename={os.path.basename(attachment_path)}"
                )

                message.attach(part)

            raw = base64.urlsafe_b64encode(message.as_bytes()).decode()

            service.users().messages().send(
                userId="me",
                body={"raw": raw}
            ).execute()

            print("📧 Alert sent successfully!")

        except Exception as e:

            print("❌ Alert error:", e)

    threading.Thread(target=email_task).start()


# -------------------------------------------------
# FIGHT DETECTION
# -------------------------------------------------

def detect_fight(frames):

    resized = [cv2.resize(f, (256,256)) for f in frames]

    t = torch.from_numpy(np.stack(resized)).float().permute(3,0,1,2).to(DEVICE)/255.0

    fast_pathway = t.unsqueeze(0)

    slow_indices = torch.linspace(0,CLIP_LENGTH-1,CLIP_LENGTH//4).long().to(DEVICE)

    slow_pathway = torch.index_select(t,1,slow_indices).unsqueeze(0)

    with torch.no_grad():

        preds = fight_detector([slow_pathway, fast_pathway])

    return torch.argmax(preds, dim=1).item()


# -------------------------------------------------
# SAVE VIDEO CLIP
# -------------------------------------------------

def save_event_clip(frames):

    if not frames:
        return None

    h,w,_ = frames[0].shape

    out = cv2.VideoWriter(
        TEMP_VIDEO_PATH,
        cv2.VideoWriter_fourcc(*'mp4v'),
        VIDEO_FPS,
        (w,h)
    )

    for f in frames:
        out.write(f)

    out.release()

    return TEMP_VIDEO_PATH


# -------------------------------------------------
# MAIN CAMERA LOOP
# -------------------------------------------------

cap = cv2.VideoCapture("https://10.186.246.24:8080/video")

frame_idx = 0

print("🚀 System Started. Press 'q' to stop.")


while True:

    ret, frame = cap.read()

    if not ret:
        break

    # 🔥 PERFORMANCE OPTIMIZATION
    small_frame = cv2.resize(frame, (640,480))

    scale_x = frame.shape[1] / 640
    scale_y = frame.shape[0] / 480

    frame_idx += 1

    events = []

    display_frame = frame.copy()


    # ---------------------------------
    # YOLO DETECTION
    # ---------------------------------

    if frame_idx % PROCESS_EVERY_N_FRAME == 0:

        weapon_results = weapon_model(small_frame, verbose=False)[0]
        fire_results = fire_model(small_frame, verbose=False)[0]

        weapon_boxes = weapon_results.boxes
        fire_boxes = fire_results.boxes

        if weapon_boxes is not None and len(weapon_boxes) > 0:
            events.append("Weapon")

        if fire_boxes is not None and len(fire_boxes) > 0:
            events.append("Fire")


    # Draw weapon boxes

    if weapon_boxes is not None:

        for box in weapon_boxes.xyxy:

            x1,y1,x2,y2 = box

            x1 = int(x1 * scale_x)
            y1 = int(y1 * scale_y)
            x2 = int(x2 * scale_x)
            y2 = int(y2 * scale_y)

            cv2.rectangle(display_frame,(x1,y1),(x2,y2),(0,0,255),2)

            cv2.putText(display_frame,"Weapon",(x1,y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,0,255),2)


    # Draw fire boxes

    if fire_boxes is not None:

        for box in fire_boxes.xyxy:

            x1,y1,x2,y2 = box

            x1 = int(x1 * scale_x)
            y1 = int(y1 * scale_y)
            x2 = int(x2 * scale_x)
            y2 = int(y2 * scale_y)

            cv2.rectangle(display_frame,(x1,y1),(x2,y2),(0,165,255),2)

            cv2.putText(display_frame,"Fire",(x1,y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,165,255),2)


    # ---------------------------------
    # FIGHT DETECTION
    # ---------------------------------

    clip_frames.append(frame)

    if len(clip_frames) == CLIP_LENGTH:

        if detect_fight(clip_frames) == 1:

            events.append("Fight")
            fight_status = True

        else:

            fight_status = False

        video_path = save_event_clip(clip_frames)

        clip_frames = []


    if fight_status:

        cv2.putText(
            display_frame,
            "WARNING: FIGHT DETECTED!",
            (40,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            3
        )


    # ---------------------------------
    # ALERT SYSTEM
    # ---------------------------------

    if events and (time.time() - last_alert_time > EVENT_COOLDOWN):

        last_alert_time = time.time()

        print("⚠ Event detected:", events)

        send_email_async(
            "🚨 Security Alert",
            events,
            video_path
        )


    # ---------------------------------
    # DISPLAY
    # ---------------------------------

    cv2.imshow("Smart Surveillance System", display_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

🔄 Loading models...
✅ All models loaded
🚀 System Started. Press 'q' to stop.
